# Shen et al. (2023) TATN — Tables II, IV, V, VI, VII

Paper-faithful Panasonic NCR18650PF temperature-transfer reproduction. The output compares only **Source pretrained** and **TATN**. Target test labels are evaluated once after training and never select a checkpoint.

The default `screening` mode cheaply produces all five table shapes and every temperature-pair comparison from two representative folds. Use `optimized_full` for all nine folds with early stopping, or `paper_full` for the released repository's fixed 2000-epoch default. Every mode has a separate Drive output and cache.

Before running, add a Colab Secret named `GITHUB_TOKEN`, grant it read access to the private repository, and enable Notebook access for the secret.

In [1]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
except ImportError:
    raise RuntimeError('This setup cell is intended for Google Colab') from None

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')

with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
    git_env.pop('GITHUB_TOKEN', None)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check the token repository permission and Notebook access') from None
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import battery_benchmark
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Package:', battery_benchmark.__file__)

Mounted at /content/drive
Repository ready: d23246f958f14fd5d81a5f68d25da5e2fe522320
Package: /content/battery-soc-transfer-benchmark/src/battery_benchmark/__init__.py


## Protocol

- Five temperatures: 25, 10, 0, -10, -20 °C; nine drive cycles each.
- Every tenth raw sample; non-overlapping windows of 90.
- Per-temperature min-max fitted to all nine cycles, exactly as stated in the paper.
- Table II: nine-fold leave-one-cycle-out per temperature; report the eight-cycle training error averaged across folds.
- Transfer source models: train on all nine source-temperature cycles.
- CNN(256×2) + four released-repo BiLSTM layers, hidden size 64, dropout 0.2.
- Adam, lr 1e-3, batch 64. The paper does not state epochs; 2000 is only the released-repo CLI default.
- Screening: Cycle_4 + UDDS folds, at most 150 epochs, minimum 30, patience 15. Diagnostic only.
- Optimized full: all nine folds, at most 300 epochs, minimum 60, patience 30.
- Semi-supervised: one test cycle, one non-test cycle with its first 50% labels, seven pseudo-labelled cycles.
- Unsupervised: one test cycle and eight pseudo-labelled cycles.

In [2]:
import torch
from battery_benchmark.utils.config import load_yaml
from battery_benchmark.reproductions.shen_2023.data import PROFILES, TEMPERATURES, load_temperature_cycles
from battery_benchmark.reproductions.shen_2023.suite import PAPER_TABLES, ShenTableSuite

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
RAW_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
RUN_MODE = 'screening'  # screening | optimized_full | paper_full
PAPER_FULL = RUN_MODE == 'paper_full'
SCREENING = RUN_MODE == 'screening'
OUTPUT_NAME = {'screening': 'tatn_screening', 'optimized_full': 'tatn_optimized_protocol', 'paper_full': 'tatn_paper_protocol'}[RUN_MODE]
OUTPUT_DIR = Path(PATHS['output_root']) / 'reproductions/shen_2023' / OUTPUT_NAME
EPOCHS = 2000 if PAPER_FULL else (150 if SCREENING else 300)
TRANSFER_EPOCHS = EPOCHS
RUN_PROFILES = ('Cycle_4', 'UDDS') if SCREENING else PROFILES
EARLY_STOPPING_PATIENCE = None if PAPER_FULL else (15 if SCREENING else 30)
MIN_EPOCHS = 0 if PAPER_FULL else (30 if SCREENING else 60)
assert RAW_ROOT.exists(), f'Missing Panasonic data: {RAW_ROOT}'
print('mode:', RUN_MODE)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('output:', OUTPUT_DIR)
print('profiles:', RUN_PROFILES)

mode: screening
device: cuda
output: /content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_screening
profiles: ('Cycle_4', 'UDDS')


In [3]:
for temperature in TEMPERATURES:
    cycles = load_temperature_cycles(RAW_ROOT, temperature)
    print(f'{temperature:>3}C:', {name: len(cycle.soc) for name, cycle in cycles.items()})

 25C: {'Cycle_1': 10965, 'Cycle_2': 11127, 'Cycle_3': 10245, 'Cycle_4': 12087, 'NN': 11699, 'US06': 4807, 'HWFET': 15178, 'LA92': 14088, 'UDDS': 22419}
 10C: {'Cycle_1': 9380, 'Cycle_2': 8110, 'Cycle_3': 10080, 'Cycle_4': 9899, 'NN': 10511, 'US06': 4200, 'HWFET': 7041, 'LA92': 12596, 'UDDS': 21047}
  0C: {'Cycle_1': 8801, 'Cycle_2': 8374, 'Cycle_3': 6247, 'Cycle_4': 7706, 'NN': 6328, 'US06': 3664, 'HWFET': 5986, 'LA92': 8267, 'UDDS': 12853}
-10C: {'Cycle_1': 6024, 'Cycle_2': 5972, 'Cycle_3': 5686, 'Cycle_4': 6111, 'NN': 5253, 'US06': 3120, 'HWFET': 5139, 'LA92': 6957, 'UDDS': 10972}
-20C: {'Cycle_1': 5072, 'Cycle_2': 5038, 'Cycle_3': 5015, 'Cycle_4': 5035, 'NN': 4537, 'US06': 2656, 'HWFET': 4232, 'LA92': 5715, 'UDDS': 8941}


In [4]:
suite = ShenTableSuite(
    RAW_ROOT, OUTPUT_DIR, epochs=EPOCHS, transfer_epochs=TRANSFER_EPOCHS,
    batch_size=64, learning_rate=1e-3, lambdas=(1.0, 1.0, 1.0),
    seed=100, device='auto', early_stopping_patience=EARLY_STOPPING_PATIENCE,
    min_epochs=MIN_EPOCHS, min_delta=1e-4,
)
table_ii = suite.run_table_ii(profiles=RUN_PROFILES)
display(table_ii.round(3).style.set_caption('Table II — source pretraining errors (%)'))
print('Paper Table II:', PAPER_TABLES['II'])
source_sanity = suite.source_sanity_table()
display(source_sanity.round(3).style.set_caption('Canonical source sanity check (%)'))
if source_sanity['MAE'].max() > 10:
    raise RuntimeError('Source-domain MAE exceeds 10%; stop before transfer and inspect preprocessing/training')

,25C,10C,0C,-10C,-20C,Avg
MAE,0.731000,1.076000,1.356000,1.857000,2.888000,1.582000
RMSE,1.029000,1.413000,1.761000,2.402000,3.691000,2.059000


Paper Table II: {'MAE': [0.323, 0.28, 0.188, 0.164, 0.515], 'RMSE': [0.404, 0.342, 0.226, 0.208, 0.652]}
[source 25C] restored source_25C.pt
[source 10C] restored source_10C.pt
[source 0C] restored source_0C.pt
[source -10C] restored source_-10C.pt
[source -20C] restored source_-20C.pt


,MAE,RMSE
temperature,,
25C,0.811000,1.072000
10C,0.993000,1.307000
0C,1.212000,1.581000
-10C,1.801000,2.334000
-20C,2.674000,3.396000


In [5]:
# Resumable: each completed fold is written to OUTPUT_DIR/cache/<protocol-id>/folds.
suite.run_all_transfers(profiles=RUN_PROFILES)
print('Transfer folds complete or restored from cache:', len(suite.records))

[semisupervised 25C->10C Cycle_4] restored
[semisupervised 25C->10C UDDS] restored
[semisupervised 25C->0C Cycle_4] restored
[semisupervised 25C->0C UDDS] restored
[semisupervised 25C->-10C Cycle_4] restored
[semisupervised 25C->-10C UDDS] restored
[semisupervised 25C->-20C Cycle_4] restored
[semisupervised 25C->-20C UDDS] restored
[semisupervised 10C->25C Cycle_4] restored
[semisupervised 10C->25C UDDS] restored
[semisupervised 10C->0C Cycle_4] restored
[semisupervised 10C->0C UDDS] restored
[semisupervised 10C->-10C Cycle_4] restored
[semisupervised 10C->-10C UDDS] restored
[semisupervised 10C->-20C Cycle_4] restored
[semisupervised 10C->-20C UDDS] restored
[semisupervised 0C->25C Cycle_4] restored
[semisupervised 0C->25C UDDS] restored
[semisupervised 0C->10C Cycle_4] restored
[semisupervised 0C->10C UDDS] restored
[semisupervised 0C->-10C Cycle_4] restored
[semisupervised 0C->-10C UDDS] restored
[semisupervised 0C->-20C Cycle_4] restored
[semisupervised 0C->-20C UDDS] restored
[sem

In [6]:
tables = suite.tables(profiles=RUN_PROFILES)
titles = {
    'II': 'Table II — fixed-temperature source pretraining',
    'IV': 'Table IV — 10C source, semi-supervised',
    'V': 'Table V — all sources, semi-supervised',
    'VI': 'Table VI — 10C source, unsupervised',
    'VII': 'Table VII — all sources, unsupervised',
}
for name in ('II', 'IV', 'V', 'VI', 'VII'):
    display(tables[name].round(3).style.set_caption(titles[name] + ' (MAE/RMSE %)'))
    print('Paper TATN reference:', PAPER_TABLES[name])
print('CSV outputs:', sorted(str(path) for path in OUTPUT_DIR.glob('table_*.csv')))

,25C,10C,0C,-10C,-20C,Avg
MAE,0.731000,1.076000,1.356000,1.857000,2.888000,1.582000
RMSE,1.029000,1.413000,1.761000,2.402000,3.691000,2.059000


Paper TATN reference: {'MAE': [0.323, 0.28, 0.188, 0.164, 0.515], 'RMSE': [0.404, 0.342, 0.226, 0.208, 0.652]}


,10C->25C,10C->0C,10C->-10C,10C->-20C,Avg
"('Source pretrained', 'MAE')",3.796000,42.267000,43.759000,48.690000,34.628000
"('Source pretrained', 'RMSE')",4.403000,44.322000,48.926000,55.542000,38.298000
"('TATN', 'MAE')",3.289000,29.338000,16.569000,18.329000,16.881000
"('TATN', 'RMSE')",3.813000,32.286000,23.336000,23.919000,20.838000


Paper TATN reference: {'MAE': [1.68, 0.466, 1.53, 2.33], 'RMSE': [2.14, 0.619, 1.94, 2.98]}


,25C->R,10C->R,0C->R,-10C->R,-20C->R,Avg
"('Source pretrained', 'MAE')",26.761000,34.628000,22.906000,25.647000,29.328000,27.854000
"('Source pretrained', 'RMSE')",29.453000,38.298000,25.173000,29.308000,33.103000,31.067000
"('TATN', 'MAE')",13.574000,16.881000,18.267000,24.049000,11.514000,16.857000
"('TATN', 'RMSE')",16.978000,20.838000,22.115000,28.713000,14.966000,20.722000


Paper TATN reference: {'MAE': [1.55, 1.5, 1.83, 1.57, 1.57], 'RMSE': [2.04, 1.92, 2.44, 2.03, 1.97]}


,10C->25C,10C->0C,10C->-10C,10C->-20C,Avg
"('Source pretrained', 'MAE')",3.796000,42.267000,43.759000,48.690000,34.628000
"('Source pretrained', 'RMSE')",4.403000,44.322000,48.926000,55.542000,38.298000
"('TATN', 'MAE')",3.626000,42.172000,50.500000,50.599000,36.724000
"('TATN', 'RMSE')",4.251000,44.299000,58.268000,58.628000,41.362000


Paper TATN reference: {'MAE': [8.01, 1.15, 2.26, 4.03], 'RMSE': [14.9, 1.5, 2.91, 5.01]}


,25C->R,10C->R,0C->R,-10C->R,-20C->R,Avg
"('Source pretrained', 'MAE')",26.761000,34.628000,22.906000,25.647000,29.328000,27.854000
"('Source pretrained', 'RMSE')",29.453000,38.298000,25.173000,29.308000,33.102000,31.067000
"('TATN', 'MAE')",30.941000,36.724000,38.755000,36.893000,34.020000,35.467000
"('TATN', 'RMSE')",34.704000,41.362000,42.049000,41.671000,38.794000,39.716000


Paper TATN reference: {'MAE': [2.63, 3.86, 4.26, 3.54, 4.31], 'RMSE': [3.44, 6.08, 6.6, 5.23, 6.01]}
CSV outputs: ['/content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_screening/table_ii.csv', '/content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_screening/table_iv.csv', '/content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_screening/table_v.csv', '/content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_screening/table_vi.csv', '/content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_screening/table_vii.csv']


## Reading the result

All modes output Tables II/IV/V/VI/VII and compare only `Source pretrained` with `TATN`. Screening keeps every temperature pair but averages two folds, so it is a directional check rather than a paper reproduction.

## Optional Table VI tensor-layout A/B

The paper describes each 90-step input as chronological `(V, I, T)` vectors. The released repository instead directly reshapes an `(N, 3)` array to `(batch, 3, 90)`, which interleaves features and time. This diagnostic changes only that reshape, keeps all training settings identical, and runs the two screening folds for Table VI. It is an implementation ablation, not an additional paper result.

In [7]:
import numpy as np
import pandas as pd

AB_ROOT = Path(PATHS['output_root']) / 'reproductions/shen_2023/tatn_layout_ablation'
released_suite = ShenTableSuite(
    RAW_ROOT, AB_ROOT / 'released_repo', epochs=EPOCHS, transfer_epochs=TRANSFER_EPOCHS,
    batch_size=64, learning_rate=1e-3, lambdas=(1.0, 1.0, 1.0),
    seed=100, device='auto', early_stopping_patience=EARLY_STOPPING_PATIENCE,
    min_epochs=MIN_EPOCHS, min_delta=1e-4, tensor_layout='released_repo',
)
for target in TEMPERATURES:
    if target != 10:
        released_suite.run_transfer(10, target, 'unsupervised', profiles=RUN_PROFILES)

layout_table_vi = pd.concat(
    {
        'paper': suite.table_for_source(10, 'unsupervised'),
        'released_repo': released_suite.table_for_source(10, 'unsupervised'),
    },
    names=['tensor_layout'],
)
layout_table_vi.to_csv(AB_ROOT / 'table_vi_layout_comparison.csv')
display(layout_table_vi.round(3).style.set_caption('Table VI tensor-layout A/B (MAE/RMSE %)'))
print('Paper TATN reference:', PAPER_TABLES['VI'])
print('saved:', AB_ROOT / 'table_vi_layout_comparison.csv')

[source 10C] training, max 150 epochs
[unsupervised 10C->25C Cycle_4] training
  completed in 30 epoch(s)
[unsupervised 10C->25C UDDS] training
  completed in 38 epoch(s)
[unsupervised 10C->0C Cycle_4] training
  completed in 0 epoch(s)
[unsupervised 10C->0C UDDS] training
  completed in 0 epoch(s)
[unsupervised 10C->-10C Cycle_4] training
  completed in 30 epoch(s)
[unsupervised 10C->-10C UDDS] training
  completed in 30 epoch(s)
[unsupervised 10C->-20C Cycle_4] training
  completed in 0 epoch(s)
[unsupervised 10C->-20C UDDS] training
  completed in 0 epoch(s)


Paper TATN reference: {'MAE': [8.01, 1.15, 2.26, 4.03], 'RMSE': [14.9, 1.5, 2.91, 5.01]}
saved: /content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_layout_ablation/table_vi_layout_comparison.csv


In [8]:
from battery_benchmark.reproductions.shen_2023.data import selected_windows
from battery_benchmark.reproductions.shen_2023.train import _cycle_prediction, predict

def layout_diagnostics(name, candidate):
    model = candidate.source_model(10)
    source_truth, source_estimate = predict(
        model, list(candidate.cycles(10).values()), candidate.device, candidate.tensor_layout
    )
    rows = []
    for target in TEMPERATURES:
        if target == 10:
            continue
        cycles = candidate.cycles(target)
        for test_profile in RUN_PROFILES:
            truths, estimates, selected, available = [], [], 0, 0
            for profile in PROFILES:
                if profile == test_profile:
                    continue
                cycle = cycles[profile]
                pseudo = _cycle_prediction(model, cycle, candidate.device, candidate.tensor_layout)
                truths.append(cycle.soc[:len(pseudo)])
                estimates.append(pseudo)
                x, _ = selected_windows(cycle, pseudo, tensor_layout=candidate.tensor_layout)
                selected += len(x)
                available += len(pseudo) // 90
            truth, estimate = np.concatenate(truths), np.concatenate(estimates)
            rows.append({
                'layout': name, 'target': f'10C->{target}C', 'fold': test_profile,
                'source_10C_MAE': np.mean(np.abs(source_truth - source_estimate)) * 100,
                'raw_pseudo_MAE': np.mean(np.abs(truth - estimate)) * 100,
                'raw_pseudo_RMSE': np.sqrt(np.mean((truth - estimate) ** 2)) * 100,
                'selected_windows': selected,
                'coverage': selected / max(available, 1),
            })
    return rows

diagnostics = pd.DataFrame(
    layout_diagnostics('paper', suite)
    + layout_diagnostics('released_repo', released_suite)
)
diagnostic_summary = diagnostics.groupby(['layout', 'target']).agg(
    source_10C_MAE=('source_10C_MAE', 'mean'),
    raw_pseudo_MAE=('raw_pseudo_MAE', 'mean'),
    raw_pseudo_RMSE=('raw_pseudo_RMSE', 'mean'),
    selected_windows=('selected_windows', 'mean'),
    coverage=('coverage', 'mean'),
)
diagnostics.to_csv(AB_ROOT / 'pseudo_layout_diagnostics_folds.csv', index=False)
diagnostic_summary.to_csv(AB_ROOT / 'pseudo_layout_diagnostics_summary.csv')
display(diagnostic_summary.round(3).style.format({'coverage': '{:.1%}'}).set_caption(
    'True target labels are used only for offline pseudo-label diagnosis'
))
print('saved:', AB_ROOT)

saved: /content/drive/MyDrive/battery_soc_benchmark/outputs/reproductions/shen_2023/tatn_layout_ablation


## Optional official `pre-10.pt` checkpoint test

This test pins the exact checkpoint published in `TL-UESTC/TATN`, verifies its SHA-256, uses the released-repository tensor layout, and reruns the Table VI screening folds. It retains this reproduction's strict pseudo-label segmentation and corrected transfer loop, thereby isolating the source checkpoint.

In [ ]:
import hashlib
import urllib.request
from battery_benchmark.reproductions.shen_2023.train import load_released_source

OFFICIAL_URL = 'https://raw.githubusercontent.com/TL-UESTC/TATN/main/models/pre-10.pt'
OFFICIAL_SHA256 = '6e152435fd5808ba56bec1237e86042799c410aa80d2f49ac99203944d3de95e'
OFFICIAL_DIR = AB_ROOT / 'official_pre10'
OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
OFFICIAL_CHECKPOINT = OFFICIAL_DIR / 'pre-10.pt'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not OFFICIAL_CHECKPOINT.exists():
    temporary = OFFICIAL_CHECKPOINT.with_suffix('.pt.part')
    urllib.request.urlretrieve(OFFICIAL_URL, temporary)
    if sha256(temporary) != OFFICIAL_SHA256:
        raise RuntimeError('Official pre-10.pt SHA-256 mismatch')
    temporary.replace(OFFICIAL_CHECKPOINT)
if sha256(OFFICIAL_CHECKPOINT) != OFFICIAL_SHA256:
    raise RuntimeError(f'Unexpected checkpoint already exists: {OFFICIAL_CHECKPOINT}')

official_suite = ShenTableSuite(
    RAW_ROOT, OFFICIAL_DIR / 'screening', epochs=EPOCHS, transfer_epochs=TRANSFER_EPOCHS,
    batch_size=64, learning_rate=1e-3, lambdas=(1.0, 1.0, 1.0),
    seed=100, device='auto', early_stopping_patience=EARLY_STOPPING_PATIENCE,
    min_epochs=MIN_EPOCHS, min_delta=1e-4, tensor_layout='released_repo',
)
official_suite._sources[10] = load_released_source(
    OFFICIAL_CHECKPOINT, official_suite.device
)
for target in TEMPERATURES:
    if target != 10:
        official_suite.run_transfer(10, target, 'unsupervised', profiles=RUN_PROFILES)

official_table_vi = official_suite.table_for_source(10, 'unsupervised')
checkpoint_table_vi = pd.concat(
    {
        'paper_retrained': suite.table_for_source(10, 'unsupervised'),
        'released_retrained': released_suite.table_for_source(10, 'unsupervised'),
        'official_pre10': official_table_vi,
    },
    names=['source_checkpoint'],
)
checkpoint_table_vi.to_csv(AB_ROOT / 'table_vi_checkpoint_comparison.csv')
display(checkpoint_table_vi.round(3).style.set_caption(
    'Table VI source-checkpoint comparison (MAE/RMSE %)'
))
print('Paper TATN reference:', PAPER_TABLES['VI'])
print('checkpoint SHA-256:', OFFICIAL_SHA256)
print('saved:', AB_ROOT / 'table_vi_checkpoint_comparison.csv')

In [ ]:
official_diagnostics = pd.DataFrame(layout_diagnostics('official_pre10', official_suite))
official_diagnostic_summary = official_diagnostics.groupby(['layout', 'target']).agg(
    source_10C_MAE=('source_10C_MAE', 'mean'),
    raw_pseudo_MAE=('raw_pseudo_MAE', 'mean'),
    raw_pseudo_RMSE=('raw_pseudo_RMSE', 'mean'),
    selected_windows=('selected_windows', 'mean'),
    coverage=('coverage', 'mean'),
)
official_diagnostics.to_csv(AB_ROOT / 'official_pre10_pseudo_diagnostics_folds.csv', index=False)
official_diagnostic_summary.to_csv(AB_ROOT / 'official_pre10_pseudo_diagnostics_summary.csv')
display(official_diagnostic_summary.round(3).style.format({'coverage': '{:.1%}'}).set_caption(
    'Official pre-10.pt; true target labels are used only for offline diagnosis'
))
print('saved:', AB_ROOT)